# Analyse the Air Traffic complaint decisions and orders

Data Source: https://portail-portal.otc-cta.gc.ca/fr/liste-decisions-8514

**Author**: Letitia Djinde Abanda

**Tools**: Python, Pandas, Numpy, SQLite


## Description of the dataset



**COLUMNS**:                            **DESCRIPTION**     

**decisionNumber**:                   Number of the decision

**airlineCode**:Code identifying the airline

**flightNumber**:Number of the flight

**flightDate**:Date when the flight occurred

**completeDate**:Date when the decision was made by the airline

**issueTierId**:ID of the type of complaints

**issue**:Type of passenger complaint

**carrierControlId**:Id of the carrier  control

**carrierControl**:Indicates whether the issue was within the airline's control

**orderStatementId**:ID of the decision

**orderStatement**:Decision made by the ailine

**itinerary**:The specific path of the flight

**Note:** Complaints that include multiple issues (for example a flight delay AND lost baggage) or multiple outcomes requested (such as compensation for inconvenience, refund and reimbursement of expenses), may have more than one order in a decision.


In [49]:
## We verify if the environment is working 

import pandas as pd
import numpy as np
import sklearn
import requests

print("Pandas version:", pd.__version__)
print("Jupyter is ready!")

Pandas version: 2.0.3
Jupyter is ready!


In [50]:
## We retrieve the data from the website : https://portail-portal.otc-cta.gc.ca/fr/liste-decisions-8514.
## The dataset was in JSON

import requests

url = "https://portail-portal.otc-cta.gc.ca/api/CRO8514Rulings"

response = requests.get(url, timeout=60)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print(response.text[:500])

Status code: 200
Content type: text/plain; charset=utf-8
[
  {
    "decisionNumber": "343353-CO-2026",
    "caseNumber": null,
    "flightInfoLong": "F8-F88705-2023-06-09-YEG-YVR",
    "airlineCode": "F8",
    "flightNumber": "F88705",
    "flightDate": "2023-06-09T00:00:00",
    "flightDateString": "2023-06-09",
    "dateCompleted": "2025-12-30T23:26:57",
    "completeDateString": "2025-12-30",
    "issueTier1Id": 1,
    "issueTier1En": "Flight Disruption",
    "issueTier1Fr": "Perturbation de vol",
    "carrierControlId": 2,
    "carr


In [51]:
import json

data = response.json()

print(type(data))

<class 'list'>


In [52]:
## We inspect and display the dataset in a table
if isinstance(data, list):
    df = pd.json_normalize(data)

elif isinstance(data, dict):
    print("Available keys:", list(data.keys()))

    if isinstance(data.get("data"), list):
        df = pd.json_normalize(data["data"])
    else:
        df = pd.json_normalize(data)

else:
    raise ValueError("Unexpected JSON structure")

print("Rows and columns:", df.shape)

display(df.head())

Rows and columns: (71319, 19)


,decisionNumber,caseNumber,flightInfoLong,airlineCode,flightNumber,flightDate,flightDateString,dateCompleted,completeDateString,issueTier1Id,issueTier1En,issueTier1Fr,carrierControlId,carrierControlEn,carrierControlFr,orderStatementId,orderStatementEn,orderStatementFr,isPublic
0,343353-CO-2026,None,F8-F88705-2023-06-09-YEG-YVR,F8,F88705,2023-06-09T00:00:00,2023-06-09,2025-12-30T23:26:57,2025-12-30,1,Flight Disruption,Perturbation de vol,2.0,Within carrier's control,Attribuable au transporteur,2,Order to refund,Ordonnance de remboursement,Y
1,343410-CO-2026,None,AC-AC105-2023-07-05-YYZ-YVR,AC,AC105,2023-07-05T00:00:00,2023-07-05,2026-01-13T09:34:46,2026-01-13,1,Flight Disruption,Perturbation de vol,2.0,Within carrier's control,Attribuable au transporteur,4,Order to compensate for inconvenience,Ordonnance d’indemnisation pour inconvénients,Y
2,343426-CO-2026,None,AC-AC1300-2022-12-18-YVR-EZE,AC,AC1300,2022-12-18T00:00:00,2022-12-18,2026-02-27T09:20:04,2026-02-27,1,Flight Disruption,Perturbation de vol,1.0,Outside of carrier's control,Indépendant de la volonté du transporteur,1,Dismissed,Rejeté,Y
3,343450-CO-2026,None,F8-F8801-2023-05-21-YYC-YVR,F8,F8801,2023-05-21T00:00:00,2023-05-21,2026-03-04T12:12:05,2026-03-04,1,Flight Disruption,Perturbation de vol,2.0,Within carrier's control,Attribuable au transporteur,4,Order to compensate for inconvenience,Ordonnance d’indemnisation pour inconvénients,Y
4,343460-CO-2026,None,AC-AC1095-2023-06-28-YYZ-SAN,AC,AC1095,2023-06-28T00:00:00,2023-06-28,2026-02-23T15:22:46,2026-02-23,1,Flight Disruption,Perturbation de vol,3.0,Within Carrier's Control but required for safety,"Attribuable au transporteur, mais nécessaire p...",1,Dismissed,Rejeté,Y


In [53]:
df.to_csv("cta_decisions.csv",
          index=False,
          encoding="utf-8-sig")

print("CSV file created successfully!")

CSV file created successfully!


In [54]:
import os
print(os.getcwd())

C:\Users\Letitia\AIRPORT PROJECT


In [55]:
df = pd.DataFrame(df)
print(type(df))

<class 'pandas.core.frame.DataFrame'>


In [56]:
len(df)
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71319 entries, 0 to 71318
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   decisionNumber      71319 non-null  object 
 1   caseNumber          0 non-null      object 
 2   flightInfoLong      71319 non-null  object 
 3   airlineCode         71319 non-null  object 
 4   flightNumber        71319 non-null  object 
 5   flightDate          71319 non-null  object 
 6   flightDateString    71319 non-null  object 
 7   dateCompleted       71319 non-null  object 
 8   completeDateString  71319 non-null  object 
 9   issueTier1Id        71319 non-null  int64  
 10  issueTier1En        71319 non-null  object 
 11  issueTier1Fr        71319 non-null  object 
 12  carrierControlId    62325 non-null  float64
 13  carrierControlEn    62325 non-null  object 
 14  carrierControlFr    62325 non-null  object 
 15  orderStatementId    71319 non-null  int64  
 16  orde

In [57]:
import os



print("Current folder:", os.getcwd())
print("Files in this folder:", os.listdir())

Current folder: C:\Users\Letitia\AIRPORT PROJECT
Files in this folder: ['.ipynb_checkpoints', 'airtravel_complaint_decisions_cleaned.ipynb', 'airtravel_complaint_decisions_db.sql', 'cta_decisions.csv']


### Data Cleaning 

In [58]:
# Drop of unecessary and duplicated columns in french as our analysis is in English

df= df.drop(['caseNumber','flightDate','dateCompleted','issueTier1Fr', 'carrierControlFr','orderStatementFr'], axis=1)

In [59]:
print(df.columns.tolist())

display(df.head())

['decisionNumber', 'flightInfoLong', 'airlineCode', 'flightNumber', 'flightDateString', 'completeDateString', 'issueTier1Id', 'issueTier1En', 'carrierControlId', 'carrierControlEn', 'orderStatementId', 'orderStatementEn', 'isPublic']


,decisionNumber,flightInfoLong,airlineCode,flightNumber,flightDateString,completeDateString,issueTier1Id,issueTier1En,carrierControlId,carrierControlEn,orderStatementId,orderStatementEn,isPublic
0,343353-CO-2026,F8-F88705-2023-06-09-YEG-YVR,F8,F88705,2023-06-09,2025-12-30,1,Flight Disruption,2.0,Within carrier's control,2,Order to refund,Y
1,343410-CO-2026,AC-AC105-2023-07-05-YYZ-YVR,AC,AC105,2023-07-05,2026-01-13,1,Flight Disruption,2.0,Within carrier's control,4,Order to compensate for inconvenience,Y
2,343426-CO-2026,AC-AC1300-2022-12-18-YVR-EZE,AC,AC1300,2022-12-18,2026-02-27,1,Flight Disruption,1.0,Outside of carrier's control,1,Dismissed,Y
3,343450-CO-2026,F8-F8801-2023-05-21-YYC-YVR,F8,F8801,2023-05-21,2026-03-04,1,Flight Disruption,2.0,Within carrier's control,4,Order to compensate for inconvenience,Y
4,343460-CO-2026,AC-AC1095-2023-06-28-YYZ-SAN,AC,AC1095,2023-06-28,2026-02-23,1,Flight Disruption,3.0,Within Carrier's Control but required for safety,1,Dismissed,Y


In [60]:
#Data type
df.dtypes

decisionNumber         object
flightInfoLong         object
airlineCode            object
flightNumber           object
flightDateString       object
completeDateString     object
issueTier1Id            int64
issueTier1En           object
carrierControlId      float64
carrierControlEn       object
orderStatementId        int64
orderStatementEn       object
isPublic               object
dtype: object

In [61]:
#Extraction of itinerary from the column: 'flightInfoLong'

df['itinerary'] = df['flightInfoLong'].str.extract(r'([A-Z]{3}-[A-Z]{3})$')

In [62]:
print(df.columns.tolist())

['decisionNumber', 'flightInfoLong', 'airlineCode', 'flightNumber', 'flightDateString', 'completeDateString', 'issueTier1Id', 'issueTier1En', 'carrierControlId', 'carrierControlEn', 'orderStatementId', 'orderStatementEn', 'isPublic', 'itinerary']


In [63]:
# Drop of the column flightInfoLong, issueTier1Id, carrierControlid, orderStatementId 
#because we dont need them for our an analysis

df = df.drop(['flightInfoLong', 'isPublic' ], axis=1)

In [64]:
print(df.columns.tolist())

['decisionNumber', 'airlineCode', 'flightNumber', 'flightDateString', 'completeDateString', 'issueTier1Id', 'issueTier1En', 'carrierControlId', 'carrierControlEn', 'orderStatementId', 'orderStatementEn', 'itinerary']


In [65]:
#Conversion of data type.

df['flightDateString'] = pd.to_datetime(df['flightDateString'], errors='coerce')
df['completeDateString'] = pd.to_datetime(df['completeDateString'], errors='coerce')
                                          
df.dtypes            

decisionNumber                object
airlineCode                   object
flightNumber                  object
flightDateString      datetime64[ns]
completeDateString    datetime64[ns]
issueTier1Id                   int64
issueTier1En                  object
carrierControlId             float64
carrierControlEn              object
orderStatementId               int64
orderStatementEn              object
itinerary                     object
dtype: object

In [68]:
df.head()

,decisionNumber,airlineCode,flightNumber,flightDateString,completeDateString,issueTier1Id,issueTier1En,carrierControlId,carrierControlEn,orderStatementId,orderStatementEn,itinerary
0,343353-CO-2026,F8,F88705,2023-06-09,2025-12-30,1,Flight Disruption,2.0,Within carrier's control,2,Order to refund,YEG-YVR
1,343410-CO-2026,AC,AC105,2023-07-05,2026-01-13,1,Flight Disruption,2.0,Within carrier's control,4,Order to compensate for inconvenience,YYZ-YVR
2,343426-CO-2026,AC,AC1300,2022-12-18,2026-02-27,1,Flight Disruption,1.0,Outside of carrier's control,1,Dismissed,YVR-EZE
3,343450-CO-2026,F8,F8801,2023-05-21,2026-03-04,1,Flight Disruption,2.0,Within carrier's control,4,Order to compensate for inconvenience,YYC-YVR
4,343460-CO-2026,AC,AC1095,2023-06-28,2026-02-23,1,Flight Disruption,3.0,Within Carrier's Control but required for safety,1,Dismissed,YYZ-SAN


In [85]:
# Mofify name of some columns names
df = df.rename(columns= {'flightDateString':'flightDate', 'completeDateString':'completeDate',
                'issueTier1En':'issue','carrierControlEn': 'carrierControl','issueTier1Id':'issueTierId', 
                         'orderStatementEn' : 'orderStatement'})

df.head()

,decisionNumber,airlineCode,flightNumber,flightDate,completeDate,issueTierId,issue,carrierControlId,carrierControl,orderStatementId,orderStatement,itinerary
0,343353-CO-2026,F8,F88705,2023-06-09,2025-12-30,1,Flight Disruption,2.0,Within carrier's control,2,Order to refund,YEG-YVR
1,343410-CO-2026,AC,AC105,2023-07-05,2026-01-13,1,Flight Disruption,2.0,Within carrier's control,4,Order to compensate for inconvenience,YYZ-YVR
2,343426-CO-2026,AC,AC1300,2022-12-18,2026-02-27,1,Flight Disruption,1.0,Outside of carrier's control,1,Dismissed,YVR-EZE
3,343450-CO-2026,F8,F8801,2023-05-21,2026-03-04,1,Flight Disruption,2.0,Within carrier's control,4,Order to compensate for inconvenience,YYC-YVR
4,343460-CO-2026,AC,AC1095,2023-06-28,2026-02-23,1,Flight Disruption,3.0,Within Carrier's Control but required for safety,1,Dismissed,YYZ-SAN


In [86]:
#Locate missing values
missing_values = df.isnull().sum()
print('Missing values per rows:\n', missing_values )

Missing values per rows:
 decisionNumber      0
airlineCode         0
flightNumber        0
flightDate          0
completeDate        0
issueTierId         0
issue               0
carrierControlId    0
carrierControl      0
orderStatementId    0
orderStatement      0
itinerary           0
dtype: int64


In [87]:
#Replace missing values in carrierControl
df['carrierControl'] = df['carrierControl'].fillna('None')
df['carrierControlId'] = df['carrierControlId'].fillna('None')
df.isnull().sum()

decisionNumber      0
airlineCode         0
flightNumber        0
flightDate          0
completeDate        0
issueTierId         0
issue               0
carrierControlId    0
carrierControl      0
orderStatementId    0
orderStatement      0
itinerary           0
dtype: int64

In [88]:
# Display the row with the missing value in flightDate

df[df['flightDate'].isna()]


,decisionNumber,airlineCode,flightNumber,flightDate,completeDate,issueTierId,issue,carrierControlId,carrierControl,orderStatementId,orderStatement,itinerary


In [89]:
#Drop of the missing value because there is a type on the website also. Date not available for that case
df = df.dropna(subset=['flightDate'])
df.isnull().sum()

decisionNumber      0
airlineCode         0
flightNumber        0
flightDate          0
completeDate        0
issueTierId         0
issue               0
carrierControlId    0
carrierControl      0
orderStatementId    0
orderStatement      0
itinerary           0
dtype: int64

In [90]:
print('Duplicate rows:' + str(df.duplicated(keep=False).sum()))
print('\n')
print(df[df.duplicated()])

Duplicate rows:0


Empty DataFrame
Columns: [decisionNumber, airlineCode, flightNumber, flightDate, completeDate, issueTierId, issue, carrierControlId, carrierControl, orderStatementId, orderStatement, itinerary]
Index: []


**Note: Complaints that include multiple issues (for example a flight delay AND lost baggage) or multiple outcomes 
requested (such as compensation for inconvenience, refund and reimbursement of expenses), may have more than one order in a decision.**

In [91]:
#Duplicates values sort by decisionNumber, issue, orderStatementEn
duplicates_rows = df[df.duplicated(subset=['decisionNumber', 'issue', 'orderStatement'],
            keep=False)].sort_values(['decisionNumber', 'issue', 'orderStatement'])

display(duplicates_rows)

,decisionNumber,airlineCode,flightNumber,flightDate,completeDate,issueTierId,issue,carrierControlId,carrierControl,orderStatementId,orderStatement,itinerary
58770,306284-CO-2026,PD,PD102,2023-03-22,2026-03-09,1,Flight Disruption,2.0,Within carrier's control,1,Dismissed,YUL-YYZ
63175,306284-CO-2026,PD,PD360,2023-03-27,2026-03-09,1,Flight Disruption,2.0,Within carrier's control,1,Dismissed,YYC-YYZ
53390,306305-CO-2025,TK,TK75,2022-10-08,2025-08-29,1,Flight Disruption,1.0,Outside of carrier's control,1,Dismissed,SAW-YVR
54943,306305-CO-2025,TK,TK1022,2022-10-08,2025-08-29,1,Flight Disruption,1.0,Outside of carrier's control,1,Dismissed,SJJ-SAW
53687,306457-CO-2025,F8,F8304,2022-12-28,2025-07-29,1,Flight Disruption,2.0,Within carrier's control,1,Dismissed,YHZ-YOW
...,...,...,...,...,...,...,...,...,...,...,...,...
54194,496746-CO-2026,WS,WS1864,2022-12-19,2026-02-05,1,Flight Disruption,1.0,Outside of carrier's control,3,Order to compensate for expenses,YVR-HNL
62232,540415-CO-2026,WS,WS209,2024-01-12,2026-05-21,1,Flight Disruption,2.0,Within carrier's control,4,Order to compensate for inconvenience,YYC-YYJ
62233,540415-CO-2026,WS,WS227,2024-01-12,2026-05-21,1,Flight Disruption,2.0,Within carrier's control,4,Order to compensate for inconvenience,YYC-YYJ
68947,560238-CO-2026,WG,WG522,2024-02-09,2026-07-24,1,Flight Disruption,3.0,Within Carrier's Control but required for safety,1,Dismissed,PVR-YYZ


We notice that there are 15671 rows with same 'decisionNumber', 'issue', 'orderStatement' and 12722 exact duplicated values

In [92]:
#Drop Dupicated values
df = df.drop_duplicates() 
print(df.duplicated().sum())

0


In [93]:
#Verification of the Data Types
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 63841 entries, 0 to 71318
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   decisionNumber    63841 non-null  object        
 1   airlineCode       63841 non-null  object        
 2   flightNumber      63841 non-null  object        
 3   flightDate        63841 non-null  datetime64[ns]
 4   completeDate      63841 non-null  datetime64[ns]
 5   issueTierId       63841 non-null  int64         
 6   issue             63841 non-null  object        
 7   carrierControlId  63841 non-null  object        
 8   carrierControl    63841 non-null  object        
 9   orderStatementId  63841 non-null  int64         
 10  orderStatement    63841 non-null  object        
 11  itinerary         63841 non-null  object        
dtypes: datetime64[ns](2), int64(2), object(8)
memory usage: 6.3+ MB


#### Document data quality

In [95]:
print("Rows after cleaning:", len(df))
print("Missing values by column:")
print(df.isna().sum())

Rows after cleaning: 63841
Missing values by column:
decisionNumber      0
airlineCode         0
flightNumber        0
flightDate          0
completeDate        0
issueTierId         0
issue               0
carrierControlId    0
carrierControl      0
orderStatementId    0
orderStatement      0
itinerary           0
dtype: int64


In [97]:
#Check if completedate < flightDate

time_processing_mask = df['completeDate']< df['flightDate']
print(df.loc[time_processing_mask,['decisionNumber','issue','carrierControl','orderStatement','itinerary','flightNumber', 'completeDate','flightDate']].head(20))

       decisionNumber                                    issue  \
14336  385288-CO-2024  Ticket, Reservation and Loyalty Program   
17830  395254-CO-2024                        Flight Disruption   
29757  385876-CO-2024  Ticket, Reservation and Loyalty Program   
30422  386942-CO-2024                        Flight Disruption   
58787  329004-CO-2025                        Flight Disruption   

                     carrierControl orderStatement itinerary flightNumber  \
14336                          None      Dismissed   YYZ-SCL        AV205   
17830  Outside of carrier's control      Dismissed   YUL-NCE        TS628   
29757                          None      Dismissed   BOS-YYC       WS1623   
30422      Within carrier's control      Dismissed   YUL-YYZ        AC483   
58787  Outside of carrier's control      Dismissed   DTW-YYZ       AC8770   

      completeDate flightDate  
14336   2024-09-16 2024-10-05  
17830   2024-05-27 2024-06-25  
29757   2024-06-11 2024-06-29  
30422   2024

**We notice that we have 6 records with CompleteDate of the decision in the past and flightdate after the decision. In cases where the issues are : Flight Disruption and Ticket, Reservation and Loyalty Program we will leave the dates as they are because it can happen that the flight disruption occurs before the flightdate due to the weather or to technical issues. Same for issues related to Ticket, Reservation and Loyalty program that can also happen before the flight. The only row that we are going to modify is the one with issue = Baggage, because we assume that the issue happend after the flight and maybe the passenger lost baggage or received a damaged baggage. In this case we wil switch the completeDate with the flightDate.** 


In [98]:
#We switch the dates between completeDate and flightDate for decision N* 329004-CO-2025

df.loc[(df['decisionNumber'] == '329004-CO-2025') & 
       (df['issue'] == 'Baggage') & (df['completeDate'] == '2025-09-17') & (df['flightDate'] == '2025-12-23'),
       ['completeDate', 'flightDate']] = ['2025-12-23','2025-09-17']


In [100]:
# Verification that we succesfuly swith the dates

print(df.loc[(df['decisionNumber'] == '329004-CO-2025') & (df['issue'] == 'Baggage')])

       decisionNumber airlineCode flightNumber flightDate completeDate  \
56927  329004-CO-2025          AC       AC8770 2025-09-17   2025-12-23   

       issueTierId    issue carrierControlId carrierControl  orderStatementId  \
56927            2  Baggage             None           None                 3   

                         orderStatement itinerary  
56927  Order to compensate for expenses   DTW-YYZ  


In [101]:
print(df.shape)
display(df.head())
print(df.dtypes)

(63841, 12)


,decisionNumber,airlineCode,flightNumber,flightDate,completeDate,issueTierId,issue,carrierControlId,carrierControl,orderStatementId,orderStatement,itinerary
0,343353-CO-2026,F8,F88705,2023-06-09,2025-12-30,1,Flight Disruption,2.0,Within carrier's control,2,Order to refund,YEG-YVR
1,343410-CO-2026,AC,AC105,2023-07-05,2026-01-13,1,Flight Disruption,2.0,Within carrier's control,4,Order to compensate for inconvenience,YYZ-YVR
2,343426-CO-2026,AC,AC1300,2022-12-18,2026-02-27,1,Flight Disruption,1.0,Outside of carrier's control,1,Dismissed,YVR-EZE
3,343450-CO-2026,F8,F8801,2023-05-21,2026-03-04,1,Flight Disruption,2.0,Within carrier's control,4,Order to compensate for inconvenience,YYC-YVR
4,343460-CO-2026,AC,AC1095,2023-06-28,2026-02-23,1,Flight Disruption,3.0,Within Carrier's Control but required for safety,1,Dismissed,YYZ-SAN


decisionNumber              object
airlineCode                 object
flightNumber                object
flightDate          datetime64[ns]
completeDate        datetime64[ns]
issueTierId                  int64
issue                       object
carrierControlId            object
carrierControl              object
orderStatementId             int64
orderStatement              object
itinerary                   object
dtype: object


In [83]:
# We create a communication between Python and Sql
%pip install sqlalchemy pymysql

Note: you may need to restart the kernel to use updated packages.


In [84]:
# Connect Python to MySQL. Put the password and your username after running the code. 
# The functions getpass() will hide your password while you are typing.

from sqlalchemy import create_engine, URL
from getpass import getpass

username = input("MySQL username: ")
password = getpass("MySQL password: ")

connection_url = URL.create(
    "mysql+pymysql",
    username=username,
    password=password,
    host="localhost",
    port=3306,
    database="airtravel_complaint_decisions"
)

engine = create_engine(connection_url)

with engine.connect() as connection:
    print("Python connected successfully to MySQL!")

MySQL username: root
MySQL password: ········
Python connected successfully to MySQL!


In [103]:
# Load of the DataFrame into MySQL
df.to_sql(
    name="airtravel_complaint_decisions_cleaned",
    con=engine,
    if_exists="fail",
    index=False
)

63841